Introduccion al problema

In [1]:
# %pip install gymnasium

### Windows:

method 1

In [2]:
# %pip install swig

In [3]:
# %pip install gymnasium[box2d]

method 2

In [4]:
# %pip install swig

In [5]:
# %pip install ufal.pybox2d

In [6]:
# %pip install pygame

### Linux:

In [7]:
# %pip install gymnasium[box2d]

https://gymnasium.farama.org/environments/box2d/lunar_lander/

In [8]:
import sys
sys.path.append("..")
from src.environments.lunar  import LunarLanderEnv

Tensorflow or Torch

In [9]:
#import torch
import tensorflow as tf

In [10]:
# Initialize the environment
lunar = LunarLanderEnv()
print(type(lunar.env.observation_space))
print(type(lunar.env.action_space))

<class 'gymnasium.spaces.box.Box'>
<class 'gymnasium.spaces.discrete.Discrete'>


El espacio de acciones es un valor del 0 al 3 que indica que acciones tomará el modulo lunar para esa iteración.

en concreto son las siguientes:

|value| action                        |
|-----|-------------------------------|
| 0   | do nothing                    |
| 1   | fire left orientation engine  |
| 2   | fire main engine              |
| 3   | fire right orientation engine |


In [11]:
lunar.env.action_space

Discrete(4)

El espacio de observaciones son un conjunto de valores flotantes y booleanos que indica el estado del modulo lunar.

en concreto son las siguientes:

|value| observation                               |
|-----|-------------------------------------------|
| 0   | coordenada X (float)                      |
| 1   | coordenada Y (float)                      |
| 2   | velocidad lineal X (float)                |
| 3   | velocidad lineal Y (float)                |
| 4   | Angulo en radianes desde -2π a +2π (float)|
| 5   | Velocidad angula (float)                  |
| 6   | Contacto de la pierna Izquierda (bool)    |
| 7   | Contacto de la pierna Derecha (bool)      |

In [12]:
# se muestran los valores minimos y maximos del espacio de observaciones.
lunar.env.observation_space

Box([ -2.5        -2.5       -10.        -10.         -6.2831855 -10.
  -0.         -0.       ], [ 2.5        2.5       10.        10.         6.2831855 10.
  1.         1.       ], (8,), float32)

In [13]:
observation_count = lunar.env.observation_space.shape[0] 
action_count = lunar.env.action_space.n

print(f"observations: {observation_count}, actions: {action_count}")

observations: 8, actions: 4


In [14]:
#valores minimos y maximos para las observaciones.
print(lunar.env.observation_space.low) 
print(lunar.env.observation_space.high)

[ -2.5        -2.5       -10.        -10.         -6.2831855 -10.
  -0.         -0.       ]
[ 2.5        2.5       10.        10.         6.2831855 10.
  1.         1.       ]


Sample ofrece una combinacion aleatoria del conjunto.

In [15]:
print(lunar.env.action_space.sample())  # Take a random action

2


In [16]:
print(lunar.env.observation_space.sample())  # Sample a random observation

[-1.5204477  -0.60305953 -3.75911    -9.644654   -5.4841166  -8.039512
  0.26974276  0.0192563 ]


Running a random episode.

In [17]:
def test_lunar_lander(steps_to_run_before_pause, agent, episodes=1):
    """
    Test the Lunar Lander environment with a given agent.
    
    Parameters:
    steps_to_run_before_pause (int): Number of steps to run before pausing for user input.
    agent: The agent to be tested in the environment.
    
    Returns:
    None
    """
    # Initialize the environment
    lunar = LunarLanderEnv(render_mode="human")
    
    if(agent is not None):
        # Set the agent's environment
        agent.lunar = lunar
        
    for _ in range(episodes):
        counter, score = 0, 0

        while True:
            """
            if steps_to_run_before_pause != 0 and counter % steps_to_run_before_pause == 0:
                input("Press Enter to continue...")
            """

            if(agent is not None):
                _, reward, done, action = agent.act()
                
            else:
                # Sample a random action from the action space
                action = lunar.env.action_space.sample()
            
                # Take a step in the environment
                _, reward, done = lunar.take_action(action, verbose=True)
                
            score += reward
            
            counter += 1
            
            if done:
                print(f"Episode finished, score: {score}")
                break
        if(agent is not None):
            # Reset the agent's environment for the next episode
            agent.lunar.reset()
        else:
            # Reset the environment for the next episode
            lunar.reset()
        
    # Close the environment
    lunar.close()

In [18]:
test_lunar_lander(steps_to_run_before_pause=0, agent=None, episodes=1)

Step taken: 3, New state: [ 0.01347208  1.4064848   0.68638027 -0.11137224 -0.01705299 -0.18588926
  0.          0.        ], Reward: -1.8246801089563849, Done: False
Step taken: 0, New state: [ 0.02024221  1.4033821   0.68640935 -0.13804436 -0.02634102 -0.18577781
  0.          0.        ], Reward: -1.1062324789370166, Done: False
Step taken: 0, New state: [ 0.02701273  1.3996803   0.686437   -0.16472144 -0.03562814 -0.18575951
  0.          0.        ], Reward: -1.1471194098919568, Done: False
Step taken: 1, New state: [ 0.03369207  1.3953738   0.67500246 -0.19158052 -0.04261931 -0.13983612
  0.          0.        ], Reward: 0.11299848913620394, Done: False
Step taken: 0, New state: [ 0.04037161  1.390468    0.67502177 -0.21825235 -0.04961035 -0.13983396
  0.          0.        ], Reward: -1.002942411017358, Done: False
Step taken: 3, New state: [ 0.0471426   1.384966    0.68646616 -0.2448696  -0.05888651 -0.18554044
  0.          0.        ], Reward: -2.369459708436209, Done: False


DQN

In [19]:
from src.agents.DQN import DQNAgent
lunar = LunarLanderEnv(render_mode=None)

Train

In [20]:
agent = DQNAgent(lunar)
#agent.train()

QNetwork:
 <DQN name=dqn, built=True>


Test

In [21]:
# agent with epsilon = 0.0 (no exploration)
agent = DQNAgent(lunar, epsilon=0.0)
agent.load_model("../saved_models/modelo_DQN_ModeloFinal.weights.h5")

QNetwork:
 <DQN name=dqn_2, built=True>


In [22]:
test_lunar_lander(steps_to_run_before_pause=25, agent=agent, episodes=1)

Episode finished, score: 232.30859093289223
Environment closed.


REINFORCE

In [23]:
from REINFORCE import REINFORCEAgent
lunar = LunarLanderEnv(render_mode=None)

ModuleNotFoundError: No module named 'REINFORCE'

In [ ]:
# agent = REINFORCEAgent(lunar, episodes=5000)
# agent.load_model("modelo_REINFORCE.h5")

In [ ]:
agent = REINFORCEAgent(lunar)
agent.load_model("modelo_REINFORCE.h5")

In [ ]:
test_lunar_lander(steps_to_run_before_pause=75, agent=agent, episodes=1)